# 🥇 Gold Layer — Customer Profiles
**LushProtein Analytics | Medallion Architecture**

Reads from `medallion/gold/`, aggregates order-level data to one row per customer, and writes to `medallion/gold/`.

| Gold Table | Gold Table | Key Transformations |
|---|---|---|
| `gold_customer_orders` | `gold_customer_profiles` | Aggregate order metrics, derive recency, pull acquisition metadata, extract RFM/gender tags, flag repeat purchasers & 90-day repurchase |

## 0. Setup & Paths

In [1]:
import pandas as pd
import numpy as np
import os
import re
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 80)

BASE       = Path.cwd()
SILVER_DIR = BASE / 'medallion' / 'silver'
GOLD_DIR   = BASE / 'medallion' / 'gold'
GOLD_DIR.mkdir(parents=True, exist_ok=True)

print(f"Base   : {BASE}")
print(f"Silver : {SILVER_DIR}")
print(f"Gold   : {GOLD_DIR}")

Base   : C:\Users\adoan\PyCharmMiscProject\Personal Applied
Silver : C:\Users\adoan\PyCharmMiscProject\Personal Applied\medallion\silver
Gold   : C:\Users\adoan\PyCharmMiscProject\Personal Applied\medallion\gold


---
## 1. Customer Profiles (`gold_customer_orders` → `gold_customer_profiles`)

### Step 1 — Load `gold_customer_orders` & aggregate order-level metrics per customer

In [2]:
# Load gold_customer_orders
co = pd.read_parquet(GOLD_DIR / 'gold_customer_orders.parquet')

# Aggregate order-level metrics per customer
customer_stats = co.groupby('customer_id').agg(
    first_order_date    =('processed_at', 'min'),
    last_order_date     =('processed_at', 'max'),
    total_orders        =('order_id', 'count'),
    total_revenue       =('price_total', 'sum'),
    avg_order_value     =('price_total', 'mean'),
    total_discount_amt  =('discount_amount', 'sum'),
    country_code        =('country_code', 'first'),
    city                =('city', 'first'),
).reset_index()

customer_stats['total_revenue']     = customer_stats['total_revenue'].round(2)
customer_stats['avg_order_value']   = customer_stats['avg_order_value'].round(2)
customer_stats['total_discount_amt']= customer_stats['total_discount_amt'].round(2)

### Step 2 — Derive recency (days since last order)

In [3]:
# Derive recency
reference_date = co['processed_at'].max()
customer_stats['recency_days'] = (reference_date - customer_stats['last_order_date']).dt.days

### Step 3 — Pull acquisition metadata from first order

In [4]:
# Pull acquisition metadata from first order
first_orders = co[co['is_first_order']].copy()

acquisition = first_orders[[
    'customer_id', 'channel', 'country_code',
    'discount_code', 'discount_amount',
    'utm_source', 'utm_medium', 'utm_campaign'
]].rename(columns={
    'channel'          : 'acquisition_channel',
    'country_code'     : 'acquisition_country',
    'discount_code'    : 'acquisition_discount_code',
    'discount_amount'  : 'acquisition_discount_amt',
    'utm_source'       : 'acquisition_utm_source',
    'utm_medium'       : 'acquisition_utm_medium',
    'utm_campaign'     : 'acquisition_utm_campaign',
})

customer_stats = customer_stats.merge(acquisition, on='customer_id', how='left')

### Step 4 — Extract RFM group, score & gender from order tags

In [5]:
# Extract RFM group, score, gender from Customer: Tags 
# Get the most recent Customer: Tags per customer
latest_tags = (co.sort_values('processed_at')
                 .groupby('customer_id')['customer_tags']
                 .last()
                 .reset_index())

def extract_tag(tag, pattern):
    if pd.isna(tag): return None
    match = re.search(pattern, tag)
    return match.group(1).strip() if match else None

latest_tags['rfm_group'] = latest_tags['customer_tags'].apply(
    lambda x: extract_tag(x, r'RFM Group ([^,]+)'))
latest_tags['rfm_score'] = latest_tags['customer_tags'].apply(
    lambda x: extract_tag(x, r'RFM Score ([^,]+)'))
latest_tags['gender'] = latest_tags['customer_tags'].apply(
    lambda x: extract_tag(x, r'\b(male|female|gender_null)\b'))

# Clean up _no_rfm
latest_tags['rfm_group'] = latest_tags['rfm_group'].replace('_no_rfm', None)
latest_tags['rfm_score'] = latest_tags['rfm_score'].replace('_no_rfm', None)

customer_stats = customer_stats.merge(
    latest_tags[['customer_id', 'rfm_group', 'rfm_score', 'gender']],
    on='customer_id', how='left'
)

### Step 5 — Flag repeat purchasers and discount-acquired customers

In [6]:
# Flag repeat purchasers and discount-acquired customers
customer_stats['is_repeat_customer']      = customer_stats['total_orders'] > 1
customer_stats['is_discount_acquired']    = customer_stats['acquisition_discount_code'].notna()

### Step 6 — Derive 90-day repeat purchase flag & save

In [7]:
# Derive 90-day repeat purchase flag
# For Q1 and Q2 — did customer make a 2nd purchase within 90 days of first?
second_orders = co[co['order_sequence'] == 2][['customer_id', 'processed_at']].rename(
    columns={'processed_at': 'second_order_date'})

customer_stats = customer_stats.merge(second_orders, on='customer_id', how='left')
customer_stats['days_to_second_order'] = (
    customer_stats['second_order_date'] - customer_stats['first_order_date']
).dt.days
customer_stats['repeat_purchase_90d'] = customer_stats['days_to_second_order'] <= 90

gold_customer_profiles = customer_stats.drop(columns=['second_order_date'])

print(f"gold_customer_profiles: {gold_customer_profiles.shape[0]:,} rows × {gold_customer_profiles.shape[1]} cols")
print(f"\nRepeat customers: {gold_customer_profiles['is_repeat_customer'].sum():,}")
print(f"Discount acquired: {gold_customer_profiles['is_discount_acquired'].sum():,}")
print(f"90-day repeat purchase: {gold_customer_profiles['repeat_purchase_90d'].sum():,}")
print(f"\nRFM Group breakdown:\n{gold_customer_profiles['rfm_group'].value_counts(dropna=False).to_string()}")
print(f"\nGender breakdown:\n{gold_customer_profiles['gender'].value_counts(dropna=False).to_string()}")

gold_customer_profiles.to_parquet(GOLD_DIR / 'gold_customer_profiles.parquet', index=False)
print("\n✅ Saved: gold_customer_profiles.parquet")

gold_customer_profiles: 13,885 rows × 24 cols

Repeat customers: 4,535
Discount acquired: 4,736
90-day repeat purchase: 3,030

RFM Group breakdown:
rfm_group
NaN                  4504
Ex lover             3215
Breakup              2882
About to dump you    1160
Don Juan              672
Platonic friend       513
Potential lover       271
Apprentice            270
Lover                 189
Soulmate               86
Flirting               63
New passion            60

Gender breakdown:
gender
NaN            4802
male           4556
female         3655
gender_null     872

✅ Saved: gold_customer_profiles.parquet


### RFM Group

4,504 nulls (32.4%) — customers with no RFM tag, likely newer customers not yet scored by the brand's loyalty app 

Ex lover (3,215) — lapsed high-value customers, the largest scored segment. These are customers who were once active but have since gone quiet — a major retention opportunity 

Breakup (2,882) — churned customers, second largest group 

About to dump you (1,160) — at-risk customers showing declining engagement 

Soulmate (86) — Champions, the best customers (R5F5M5). Small but highly valuable

The top 3 segments (Ex lover, Breakup, About to dump you) together account for 52% of all customers — meaning over half the customer base is either churned or at risk. 